# 01 — Platform and methodology

**Question.** Under what conditions were the numbers taken, and how were they validated?

Everything in the analysis derives from raw per-process records kept unmodified in `data/raw/`. This notebook documents the platform, how each campaign was launched, the validity rules, and how repeatable the measurements are.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench import report, roofline, plotting
from hwbench.notebook import setup
ctx = setup('01_platform_and_methodology')
thesis, ext, theory = ctx.thesis, ctx.ext, ctx.theory
table, figure, say, say_claims = ctx.table, ctx.figure, ctx.say, ctx.say_claims
from hwbench import claims

## 1. Platform record

Read on the execution node, without root, for one Flat and one Cache campaign. An unreadable source is recorded as such: for example, RAPL and MSR access is restricted, which bounds which counters can be used.

In [2]:
import json
def platform_of(mode):
    camp = sorted(ext[ext.memory_mode == mode].campaign_id.unique())[0]
    return camp, json.loads((ROOT / 'data/raw' / mode / camp / 'system/platform.json').read_text()), json.loads((ROOT / 'data/raw' / mode / camp / 'system/topology.json').read_text())
rows = []
for mode in ['flat', 'cache']:
    camp, plat, topo = platform_of(mode)
    proc, pol = plat['processor'], plat['kernel_policy']
    rows.append({
        'mode': mode, 'campaign': camp, 'node': topo['hostname'].split('.')[0], 'cpu': topo['cpu_model'],
        'sockets': topo['socket_count'], 'cores': topo['physical_cores'], 'threads/core': topo['threads_per_core'],
        'NUMA nodes': len(topo['numa_nodes']), 'kernel': plat['os']['kernel'], 'OS': plat['os']['pretty_name'],
        'cpufreq max (MHz)': proc['cpuinfo_max_khz'] / 1000, 'THP': pol.get('transparent_hugepage'),
        'perf_event_paranoid': pol.get('perf_event_paranoid'), 'NUMA balancing': pol.get('numa_balancing'),
    })
table(pd.DataFrame(rows).set_index('mode').T.reset_index().rename(columns={'index': 'property'}))

property,flat,cache
campaign,flat-bandwidth2-20261011T230637Z,cache-bandwidth2-20261011T230639Z
node,cresco8-hbm14,cresco8-hbm01
cpu,Intel(R) Xeon(R) CPU Max 9480,Intel(R) Xeon(R) CPU Max 9480
sockets,2,2
cores,112,112
threads/core,1,1
NUMA nodes,4,2
kernel,5.14.0-427.13.1.el9_4.x86_64,5.14.0-427.13.1.el9_4.x86_64
OS,Red Hat Enterprise Linux 9.4 (Plow),Red Hat Enterprise Linux 9.4 (Plow)
cpufreq max (MHz),1901.000000,1901.000000


In [3]:
camp, plat, topo = platform_of('flat')
hm = plat['hmat_access']
nodes = {n['node']: n for n in topo['numa_nodes']}
table(pd.DataFrame([dict(node=int(k), tier=nodes[int(k)]['memory_tier'], socket=nodes[int(k)]['socket'], gib=round(nodes[int(k)]['memory_gib'], 1),
        hmat_read_GBs=v['read_bandwidth'] / 1000, hmat_write_GBs=v['write_bandwidth'] / 1000,
        hmat_read_ns=v['read_latency']) for k, v in hm.items()]))

node,tier,socket,gib,hmat_read_GBs,hmat_write_GBs,hmat_read_ns
0,ddr,0,503.200000,262.100000,176.100000,91
1,ddr,1,503.900000,262.100000,176.100000,91
2,hbm,0,64.000000,869.100000,618.200000,127
3,hbm,1,64.000000,869.100000,618.200000,127


In [4]:
say_claims(claims.core_scaling(ext, thesis, ROOT) + claims.latency(ext, theory, ROOT), ['hmat_gap', 'hmat_latency'])

* The firmware (ACPI HMAT) advertises 262 GB/s (DDR) and 869 GB/s (HBM) read bandwidth per socket. Measured DDR read is 100% of the DDR figure; HBM reaches 62% of the HBM figure. The gap is attributed to concurrency and clock (above) or to the memory configuration (SNC4/HBM-only would change it); neither is tested here.
* The firmware (HMAT) advertises unloaded read latencies of 91 ns (DDR) and 127 ns (HBM); measured dependent-load latency is higher (122 and 145 ns), plausibly because it includes address translation and the cache path.

## 2. Execution rules and provenance

* **Flat** campaigns run on the single dedicated Flat node, outside Slurm, in a tmux session started over SSH. Before measuring, the runner checks the node is idle (no Slurm job placed on it, 1-minute load below 1), and every record is bracketed by load and job snapshots. A foreign job observed at any point marks the record `interference_suspected` and processing rejects it.
* **Cache** campaigns run only under Slurm, exclusively (`--exclusive --mem=0`), with the Flat node excluded in the versioned submission template; the runner aborts if the node is the Flat node or its topology is not Cache.
* The October-11 Flat campaigns were run through an exclusive Slurm allocation on the same node (`launch_method = slurm_exclusive`); they are reused unchanged, and the cross-check below compares them with SSH runs.

In [5]:
sel = pd.concat([thesis, ext])
prov = (sel.groupby(['memory_mode', 'campaign_id', 'launch_method', 'node', 'memory_mode_verified'], dropna=False)
        .agg(samples=('accepted', 'size'), source_sha256=('source_sha256', lambda s: ';'.join(sorted({x[:8] for x in s.dropna()})))).reset_index())
table(prov)

memory_mode,campaign_id,launch_method,node,memory_mode_verified,samples,source_sha256
cache,cache-bandwidth2-20261011T230639Z,slurm_exclusive,cresco8-hbm01,True,199,a0783284
cache,cache-gemm-20261012T063700Z,slurm_exclusive,cresco8-hbm01,True,515,4ca2308f
cache,cache-gemm-20261012T073817Z,slurm_exclusive,cresco8-hbm01,True,25,4ca2308f
cache,cache-hierarchy-20261012T062600Z,slurm_exclusive,cresco8-hbm01,True,265,a0783284
cache,cache-isa-20261011T225344Z,slurm_exclusive,cresco8-hbm01,True,390,a0783284
cache,cache-span-20261012T075002Z,slurm_exclusive,cresco8-hbm01,True,110,a0783284
cache,cache-thesis-20261011T182120Z,slurm_exclusive,cresco8-hbm01,True,260,327602c9;59c1de98;80cab341
cache,cache-thesis-20261011T182359Z,slurm_exclusive,cresco8-hbm01,True,230,59c1de98
cache,cache-thesis-20261011T183752Z,slurm_exclusive,cresco8-hbm01,True,125,6071308b;904e6684
cache,cache-xcheck-20261012T065205Z,slurm_exclusive,cresco8-hbm02,True,40,a0783284


## 3. Validity rules and repeatability

A sample is used only if the benchmark's own checksum validated, the process ran on the expected host with the expected topology, its clock window lies inside the process window, the disassembly of the compiled benchmark contains the intended instruction (a whole-binary check, not a check of the measured loop), GEMMs show an AMX execution in oneDNN's verbose log, and no foreign Slurm job was seen. Each configuration needs five steady samples; variation above 5% is flagged, not hidden.

In [6]:
v = ctx.validation
say(f"**Validation.** status `{v['status']}`; {v['selected_samples']} thesis samples, {v['extended_samples']} extended samples; "
    f"{len(v['errors'])} errors, {len(v['warnings'])} warnings.")
table(pd.DataFrame({'warning': v['warnings']}))

**Validation.** status `passed`; 1345 thesis samples, 3964 extended samples; 0 errors, 51 warnings.

warning
5 dual-socket samples overlap by less than 99%
"Fewer than 5 samples (4): {'campaign_id': 'cache-bandwidth2-20261011T230639Z', 'memory_mode': 'cache', 'socket_scope': 'dual', 'socket': nan, 'memory_socket': 0.0, 'memory_target': 'effective_cache', 'benchmark': 'stream_triad_nt', 'kernel': 'triad_nt', 'dtype': nan, 'm': nan, 'n': nan, 'k': 1.0, 'threads': 112, 'working_set_bytes': 103079215104.0, 'notes': 'bw2_triad_nt_dual'}"
Repeat run deviates +2.6% from October (flat/stream_triad/hbm/socket 1 on cresco8-hbm14)
Repeat run deviates +4.0% from October (cache/stream_triad/effective_cache/socket 1 on cresco8-hbm02)
Repeat run deviates +7.3% from October (cache/stream_triad/effective_cache/socket 0 on cresco8-hbm02)
"Variation above 5%: ('cache-gemm-20261012T063700Z', 'cache', 'single', 0.0, 0.0, 'effective_cache', 'onednn_gemm_sweep', 'gemm', 'bf16', 1.0, 14336.0, 4096.0, 56, 103112769536.0, 'gemm_sweep')"
"Variation above 5%: ('cache-gemm-20261012T063700Z', 'cache', 'single', 0.0, 0.0, 'effective_cache', 'onednn_gemm_sweep', 'gemm', 'bf16', 1.0, 4096.0, 14336.0, 56, 103112769536.0, 'gemm_sweep')"
"Variation above 5%: ('cache-gemm-20261012T063700Z', 'cache', 'single', 0.0, 0.0, 'effective_cache', 'onednn_gemm_sweep', 'gemm', 'bf16', 16.0, 14336.0, 4096.0, 56, 103112769536.0, 'gemm_sweep')"
"Variation above 5%: ('cache-gemm-20261012T063700Z', 'cache', 'single', 0.0, 0.0, 'effective_cache', 'onednn_gemm_sweep', 'gemm', 'bf16', 4.0, 14336.0, 4096.0, 56, 103112769536.0, 'gemm_sweep')"
"Variation above 5%: ('cache-gemm-20261012T063700Z', 'cache', 'single', 0.0, 0.0, 'effective_cache', 'onednn_gemm_sweep', 'gemm', 'bf16', 4.0, 4096.0, 14336.0, 56, 103112769536.0, 'gemm_sweep')"


In [7]:
def cv(frame, metric):
    keys = ['memory_mode', 'benchmark', 'dtype', 'socket_scope', 'socket', 'memory_target', 'threads', 'working_set_bytes', 'notes', 'campaign_id']
    g = frame.groupby(keys, dropna=False)[metric]
    return (100 * g.std() / g.mean()).dropna()
c = pd.concat([cv(ext[ext.bandwidth_gbs.notna()], 'bandwidth_gbs'), cv(ext[ext.performance_gops.notna() & ext.bandwidth_gbs.isna()], 'performance_gops')])
table(c.describe(percentiles=[.5, .9, .99]).to_frame('CV % across configurations').reset_index())

index,CV % across configurations
count,429.000000
mean,4.153000
std,12.976000
min,0.003000
50%,0.795000
90%,3.495000
99%,65.008000
max,72.225000


## 4. Repeat run against the October values

Flat repeats run over SSH on the same node; Cache repeats run on a second node, so their deviation measures node-to-node variance. The acceptance threshold is 2%.

In [8]:
x = report.xcheck_table(ext, thesis)
table(x[['memory_mode', 'benchmark', 'dtype', 'memory_target', 'socket', 'node', 'new', 'october', 'october_launch', 'deviation_percent', 'within_tolerance', 'cv_percent']])

memory_mode,benchmark,dtype,memory_target,socket,node,new,october,october_launch,deviation_percent,within_tolerance,cv_percent
cache,amx_peak,bf16,registers,0.000000,cresco8-hbm02,112599.700000,112590.280000,slurm_exclusive,0.008000,True,0.860000
cache,amx_peak,bf16,registers,1.000000,cresco8-hbm02,111995.816000,111779.390000,slurm_exclusive,0.194000,True,0.751000
cache,avx512_peak,fp32,registers,0.000000,cresco8-hbm02,8811.424000,8764.260000,slurm_exclusive,0.538000,True,1.642000
cache,avx512_peak,fp32,registers,1.000000,cresco8-hbm02,8752.005000,8724.853000,slurm_exclusive,0.311000,True,1.506000
cache,avx512_peak,fp64,registers,0.000000,cresco8-hbm02,4406.500000,4396.041000,slurm_exclusive,0.238000,True,1.612000
cache,avx512_peak,fp64,registers,1.000000,cresco8-hbm02,4378.954000,4364.778000,slurm_exclusive,0.325000,True,1.433000
cache,stream_triad,nan,effective_cache,0.000000,cresco8-hbm02,526.497000,490.710000,slurm_exclusive,7.293000,False,2.547000
cache,stream_triad,nan,effective_cache,1.000000,cresco8-hbm02,507.791000,488.266000,slurm_exclusive,3.999000,False,3.935000
flat,amx_peak,bf16,registers,0.000000,cresco8-hbm14,112875.039000,112621.186000,slurm_exclusive,0.225000,True,0.912000
flat,amx_peak,bf16,registers,1.000000,cresco8-hbm14,112594.959000,112419.402000,slurm_exclusive,0.156000,True,0.828000


In [9]:
flat_x, cache_x = x[x.memory_mode == 'flat'], x[x.memory_mode == 'cache']
say(f"**Result.** Flat, SSH versus the October Slurm-exclusive runs: {flat_x.within_tolerance.mean():.0%} of {len(flat_x)} points within 2% "
    f"(largest {flat_x.deviation_percent.abs().max():.1f}%). Cache, second node versus the first: {cache_x.within_tolerance.mean():.0%} of {len(cache_x)} within 2% "
    f"(largest {cache_x.deviation_percent.abs().max():.1f}%).")

**Result.** Flat, SSH versus the October Slurm-exclusive runs: 90% of 10 points within 2% (largest 2.6%). Cache, second node versus the first: 75% of 8 within 2% (largest 7.3%).

In [10]:
say_claims(claims.xcheck(ext, thesis))

* Repeat runs on the same Flat node differ by at most 2.6% (90% of 10 points within 2%), including the October Slurm-exclusive runs against the SSH runs, so the launch method is not a visible source of error. Cache repeats on a second node differ by up to 7.3% (75% of 8 within 2%), which is node-to-node variance. These cross-run differences are a systematic uncertainty on top of the sample-to-sample spread.

## 5. Traffic and operation conventions

* Triad counts 24 B per element (two reads, one write); read-only counts 8 B per element; operations are counted as multiply + add = 2.
* GEMM traffic is weights + input activations + output per call, with weights drawn round-robin from a pool larger than four times the last-level cache so they stream from memory.
* Effective clock is user-mode core cycles ÷ (threads × wall time), self-monitored per thread through `perf_event_open`; AMX-busy is the `EXE.AMX_BUSY` event over the same cycles.